In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

### Read Bus Stops (nx without service)

In [2]:
bus_stops_ox = gpd.read_file(r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Insumos OSMNX/Transporte Publico/Snapped/ox_nearestEdge/bus_stops_snapped_ox.shp')

In [4]:
stops_preprocessed = bus_stops_ox.copy()

In [5]:
MIN_REL_POS = 0.1
MAX_REL_POS = 0.9
FORBIDDEN_LOW = 0.45
FORBIDDEN_HIGH = 0.55
MIN_GAP = 0.002

In [6]:
left_range = (MIN_REL_POS, FORBIDDEN_LOW - MIN_GAP)
right_range = (FORBIDDEN_HIGH + MIN_GAP, MAX_REL_POS)
left_size = left_range[1] - left_range[0]
right_size = right_range[1] - right_range[0]

print(f"Left range: {left_range}, size: {left_size}")
print(f"Right range: {right_range}, size: {right_size}")

Left range: (0.1, 0.448), size: 0.348
Right range: (0.552, 0.9), size: 0.348


In [7]:
def adjust_group(group):
    group = group.sort_values('RelPosOnLi').copy()
    n = len(group)

    #if only one stop per link
    if n == 1:
        #set to 0.1 if lower, set to 0.9 if higher
        position = np.clip(group['RelPosOnLi'].values, MIN_REL_POS, MAX_REL_POS)

        #avoid forbidden zone 0.45-0.55
        if FORBIDDEN_LOW < position[0] < FORBIDDEN_HIGH:
            if position[0] < 0.5:
                position[0] = FORBIDDEN_LOW - MIN_GAP
            else:
                position[0] = FORBIDDEN_HIGH + MIN_GAP
        
        group['RelPos_New'] = position
        return group
    
    # if multiple stops per link
    left_range = (MIN_REL_POS, FORBIDDEN_LOW - MIN_GAP)
    right_range = (FORBIDDEN_HIGH + MIN_GAP, MAX_REL_POS)
    left_size = left_range[1] - left_range[0]
    right_size = right_range[1] - right_range[0]
    total_size = left_size + right_size

    # asignar proporcionalmente a cada lado
    n_left = int(np.round(n * (left_size / total_size)))
    n_right = n - n_left

    if n_left == 0:
        n_left = 1
        n_right = n - 1
    elif n_right == 0:
        n_right = 1
        n_left = n - 1

    # generar posiciones para cada lado
    left_positions = np.linspace(left_range[0], left_range[1], n_left)
    right_positions = np.linspace(right_range[0], right_range[1], n_right)
    new_positions = np.concatenate([left_positions, right_positions])

    group = group.sort_values('RelPosOnLi')
    group['RelPos_New'] = new_positions

    # asegurar gap mínimo
    pos = group['RelPos_New'].values
    for i in range(1, len(pos)):
        if pos[i] - pos[i-1] < MIN_GAP:
            pos[i] = pos[i-1] + MIN_GAP
    
    pos = np.clip(pos, MIN_REL_POS, MAX_REL_POS)
    group['RelPos_New'] = pos
    
    return group

In [8]:
stops_preprocessed = (
    stops_preprocessed
    .groupby('NearestLin', group_keys=False)
    .apply(adjust_group)
)

/var/folders/8s/00_wwq9j23b09mnd7gp105m00000gp/T/ipykernel_1404/720634648.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(adjust_group)


In [11]:
stops_preprocessed.to_file(r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Insumos OSMNX/Transporte Publico/Snapped/ox_nearestEdge/RelPos preprocess/bus_stops_ox_relpos_preprocessed.shp')

In [16]:
stops_preprocessed[stops_preprocessed['id']==0]

,id,mobiliario,iluminaci,señal_ver,señal_hor,banqueta,vegetació,ruta_1,ruta_2,ruta_3,...,NearestEdg,SnapDist_m,NearestLin,RelPosOnLi,FromNodeNo,ToNodeNo,NearestL_1,stop_id,geometry,RelPos_New
7622,0.0,0.0,1.0,1.0,1.0,1.0,0.0,Troncal 19 Periferico,T19-C05,C53,...,"(1423195343, 7703638382, 0)",4.783494,331489,0.302852,136446,179947,secondary,7623,POINT (665627.854 2293420.307),0.302852
